# 3.08 Elasticsearch Vector Store

**Elasticsearch**는 8.x부터 dense_vector 타입과 ANN 검색을 공식 지원한다. 기존 ES 스택(로그·검색)을 그대로 재활용해 RAG를 추가할 수 있다.

## 학습 목표

- `ElasticsearchStore(es_url=..., index_name=..., embedding=..., strategy=...)` 초기화
- ES DSL 필터 (`[{'term': {'metadata.source.keyword': 'a'}}]`)
- `ApproxRetrievalStrategy(hybrid=True)` 로 dense + BM25 RRF
- 점수 해석 (ES는 `_score` 가 유사도 스코어, 높을수록 가까움)

## 언제 쓰나

- 기존 ELK 스택 재사용 — 로그·검색·벡터를 한 클러스터에서
- Elastic Cloud의 관리형 활용
- 강력한 쿼리 언어(DSL)와 aggregation이 필요할 때

## 3.08.1 환경 설정 / 서비스 연결

로컬 Docker:
```bash
docker run -p 9200:9200 -e discovery.type=single-node -e xpack.security.enabled=false \
  docker.elastic.co/elasticsearch/elasticsearch:8.15.0
```

Elastic Cloud는 `.env`에 `ES_URL` + `ES_API_KEY`.

probe 셀은 `Elasticsearch(...).info()` 로 실제 접속을 확인한다.

In [ ]:
# !pip install -U langchain-elasticsearch elasticsearch langchain-openai

import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요"

from elasticsearch import Elasticsearch

ES_URL = os.environ.get("ES_URL", "http://localhost:9200")
ES_API_KEY = os.environ.get("ES_API_KEY")

if ES_API_KEY:
    es_client = Elasticsearch(ES_URL, api_key=ES_API_KEY)
else:
    es_client = Elasticsearch(ES_URL)

# 실제 health check (실패 시 ConnectionError)
info = es_client.info()
print("Elasticsearch OK:", info["version"]["number"])

## 3.08.2 기본 사용법

`ElasticsearchStore.from_documents(docs, embedding, index_name=..., es_connection=...)` 로 업서트. `strategy=ApproxRetrievalStrategy()` 가 기본 — HNSW 기반 ANN.

In [ ]:
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

docs = [
    Document(page_content="Elasticsearch 8부터 dense_vector 타입과 ANN 검색을 지원한다.", metadata={"source": "elastic", "lang": "ko"}),
    Document(page_content="BM25는 Elasticsearch의 기본 텍스트 스코어링이다.", metadata={"source": "elastic", "lang": "ko"}),
    Document(page_content="RRF combines sparse and dense retrievers with reciprocal rank.", metadata={"source": "paper", "lang": "en"}),
    Document(page_content="Kibana로 벡터 검색 결과를 대시보드화할 수 있다.", metadata={"source": "elastic", "lang": "ko"}),
]

INDEX = "vs-demo"

# 이전 실행 흔적 제거
if es_client.indices.exists(index=INDEX):
    es_client.indices.delete(index=INDEX)

store = ElasticsearchStore.from_documents(
    docs,
    embedding=embeddings,
    index_name=INDEX,
    es_connection=es_client,
)
es_client.indices.refresh(index=INDEX)

for d in store.similarity_search("벡터 검색", k=3):
    print("-", d.metadata.get("source"), "|", d.page_content[:50])

## 3.08.3 메타데이터 필터링 — ES DSL

LangChain `ElasticsearchStore` 는 `filter=` 인자에 **ES query DSL 리스트**를 그대로 받는다.

- `term`: 정확 일치 (keyword 타입)
- `terms`: IN
- `range`: 범위
- `bool`: `must` / `should` / `must_not` 조합

LangChain이 메타데이터를 `metadata.*` 경로로 저장하므로 `metadata.source.keyword` 형태로 접근 (기본 동적 매핑 기준).

In [ ]:
hits = store.similarity_search(
    "검색 엔진",
    k=5,
    filter=[{"term": {"metadata.lang.keyword": "ko"}}],
)
for d in hits:
    print("-", d.metadata, "|", d.page_content[:50])

## 3.08.4 점수 포함 · MMR

Elasticsearch `_score` 는 **유사도**(높을수록 가까움).

In [ ]:
print("--- with_score (유사도) ---")
for doc, score in store.similarity_search_with_score("벡터 검색", k=3):
    print(f"{score:.4f}  {doc.metadata.get('source')}")

print("\n--- MMR ---")
for d in store.max_marginal_relevance_search("검색 엔진", k=3, fetch_k=4, lambda_mult=0.4):
    print("-", d.metadata.get("source"))

## 3.08.5 하이브리드 검색 — RRF

`ApproxRetrievalStrategy(hybrid=True)` 는 knn(dense) + `query_string`(BM25) 결과를 **RRF**로 결합한다. ES 8.11+ 에서 네이티브 `rank: {rrf: {}}` 를 사용.

In [ ]:
from langchain_elasticsearch import ApproxRetrievalStrategy

HYBRID_INDEX = "vs-demo-hybrid"
if es_client.indices.exists(index=HYBRID_INDEX):
    es_client.indices.delete(index=HYBRID_INDEX)

hybrid_store = ElasticsearchStore.from_documents(
    docs,
    embedding=embeddings,
    index_name=HYBRID_INDEX,
    es_connection=es_client,
    strategy=ApproxRetrievalStrategy(hybrid=True),
)
es_client.indices.refresh(index=HYBRID_INDEX)

# 정확 키워드 "BM25" 를 찾을 때 dense만으로는 놓칠 수 있는 문서가 올라온다
for d in hybrid_store.similarity_search("BM25", k=3):
    print("-", d.metadata.get("source"), "|", d.page_content[:60])

## 3.08.6 정리

데모 인덱스 삭제.

In [ ]:
for ix in [INDEX, HYBRID_INDEX]:
    if es_client.indices.exists(index=ix):
        es_client.indices.delete(index=ix)
        print("deleted:", ix)

es_client.close()

## 체크리스트

- [ ] 메타데이터 필터 경로는 `metadata.<field>.keyword` (기본 동적 매핑)
- [ ] 업서트 후 **`indices.refresh`** — 검색 가능 상태로 만든다
- [ ] Hybrid RRF는 ES 8.11+ 서브스크립션(플래티넘) 기능 — 버전·라이선스 확인
- [ ] 임베딩 차원 수는 매핑의 `dims` 와 일치 — 임베딩 교체 시 재인덱싱

## 다음

- `05_retrievers/` — ES retriever · BM25Retriever
- `12_observability/` — ES 결과를 Kibana 대시보드로

## 참고

- ES vector search: https://www.elastic.co/guide/en/elasticsearch/reference/current/knn-search.html
- `langchain-elasticsearch`: https://python.langchain.com/docs/integrations/vectorstores/elasticsearch